# 02 — Silver : Nettoyage et typage

Lit la table Delta **bronze**, applique les règles de nettoyage,
et écrit le résultat en Delta **silver**.

Chaque règle est documentée avec le nombre de lignes retirées.

In [ ]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from pathlib import Path

DATA_DIR = Path("../data")
BRONZE_PATH = str(DATA_DIR / "bronze" / "flights")
SILVER_PATH = str(DATA_DIR / "silver" / "flights")

df = DeltaTable(BRONZE_PATH).to_pandas()
initial_count = len(df)
print(f"Bronze : {initial_count:,} lignes")
print(f"Colonnes : {len(df.columns)}")

In [ ]:
# Colonnes utiles à conserver
KEEP_COLS = [
    "FL_DATE", "OP_UNIQUE_CARRIER", "OP_CARRIER_AIRLINE_ID",
    "OP_CARRIER", "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_ABR",
    "DEP_DELAY", "DEP_DEL15", "ARR_DELAY", "ARR_DEL15",
    "CANCELLED", "CANCELLATION_CODE",
    "AIR_TIME", "DISTANCE",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]

available = [c for c in KEEP_COLS if c in df.columns]
df = df[available].copy()
print(f"Colonnes retenues : {len(available)}")

In [ ]:
cleaning_log = []

def log_rule(name, before, after):
    removed = before - after
    cleaning_log.append({"Règle": name, "Avant": before, "Après": after, "Retirées": removed})
    print(f"  {name}: {removed:,} lignes retirées ({before:,} → {after:,})")

print("Nettoyage en cours...\n")
count = len(df)

# Règle 1 : Supprimer les lignes sans date de vol
df = df.dropna(subset=["FL_DATE"])
log_rule("R1 — FL_DATE manquante", count, len(df))
count = len(df)

# Règle 2 : Supprimer les lignes sans aéroport d'origine ou de destination
df = df.dropna(subset=["ORIGIN", "DEST"])
log_rule("R2 — ORIGIN ou DEST manquant", count, len(df))
count = len(df)

# Règle 3 : Supprimer les lignes où le retard et le statut annulé sont tous les deux absents
mask = df["ARR_DELAY"].isna() & (df["CANCELLED"] != 1)
df = df[~mask]
log_rule("R3 — Ni retard ni annulation", count, len(df))
count = len(df)

# Règle 4 : Distance <= 0 (incohérent)
df = df[df["DISTANCE"] > 0]
log_rule("R4 — Distance <= 0", count, len(df))
count = len(df)

# Règle 5 : Doublons exacts
df = df.drop_duplicates()
log_rule("R5 — Doublons exacts", count, len(df))
count = len(df)

In [ ]:
# Résumé du nettoyage
summary = pd.DataFrame(cleaning_log)
total_removed = initial_count - len(df)
print(f"\n{'='*60}")
print(f"Total lignes retirées : {total_removed:,} / {initial_count:,} ({100*total_removed/initial_count:.2f}%)")
print(f"Lignes silver finales : {len(df):,}")
summary

In [ ]:
# Typage
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df["CANCELLED"] = df["CANCELLED"].astype(int)

float_cols = ["DEP_DELAY", "ARR_DELAY", "AIR_TIME", "DISTANCE",
              "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
              "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

str_cols = ["OP_UNIQUE_CARRIER", "OP_CARRIER", "ORIGIN", "DEST",
            "ORIGIN_CITY_NAME", "DEST_CITY_NAME",
            "ORIGIN_STATE_ABR", "DEST_STATE_ABR", "CANCELLATION_CODE"]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("Types après typage :")
df.dtypes

In [ ]:
import os
os.makedirs(SILVER_PATH, exist_ok=True)

write_deltalake(SILVER_PATH, df, mode="overwrite")
print(f"✅ Silver écrit : {len(df):,} lignes")